In [ ]:
# =========================
# Part A: LDA + Heatmap only
# =========================
# -*- coding: utf-8 -*-

import os, re, glob, warnings, pickle
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")

# ======= global fonts =======
plt.rcParams.update({
    "axes.titlesize": 30,
    "axes.labelsize": 22,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "font.size": 24
})

# ======= Config =======
NETWORK_DIR        = "network"
OUTPUT_DIR         = "result_topic_lda"
CLUSTERS: Optional[List[int]] = None

DESIRED_ORDER      = [4, 8, 7, 3, 2, 1, 5]

SELECTION_MODE     = "all"      # "all" / "topk" / "threshold" / "both"
TOP_N_PER_CLUSTER  = 10000
SCORE_THRESHOLD    = 0.99

HEATMAP_CMAP       = "OrRd"
SAVE_FIG           = True

N_TOPICS           = 10
RANDOM_STATE       = 42
LDA_MAX_ITER       = 200
LDA_LEARNING_METHOD= "batch"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======= Utils =======
def discover_clusters(network_dir: str) -> List[int]:
    paths = glob.glob(os.path.join(network_dir, "C*_network.*"))
    cids = []
    for p in paths:
        m = re.search(r"C(\d+)_network\.", os.path.basename(p))
        if m:
            cids.append(int(m.group(1)))
    return sorted(set(cids))

def load_network_for_cluster(cid: int, network_dir: str) -> pd.DataFrame:
    candidates = [
        os.path.join(network_dir, f"C{cid}_network.txt"),
        os.path.join(network_dir, f"C{cid}_network.tsv"),
    ]
    path = None
    for p in candidates:
        if os.path.exists(p):
            path = p
            break
    assert path is not None, f"Network file for C{cid} not found in {network_dir}"

    df = pd.read_csv(path, sep="\t")
    cols = {c.lower(): c for c in df.columns}
    tf_col = cols.get("tf", None)
    tg_col = cols.get("tg", None)
    sc_col = cols.get("score", None)
    assert tf_col and tg_col and sc_col, f"{path} must contain TF/TG/Score columns."

    df = df[[tf_col, tg_col, sc_col]].copy()
    df.columns = ["TF", "TG", "Score"]
    return df

def select_edges_df(df: pd.DataFrame, mode="all", top_n=4000, thr=0.99) -> pd.DataFrame:
    if df.empty:
        return df
    if mode == "all":
        return df.sort_values("Score", ascending=False).reset_index(drop=True)

    scores = df["Score"].values
    idx_all = np.arange(len(df))
    idx_topk = idx_thr = None

    if mode in ("topk", "both"):
        k = min(top_n, len(df))
        part = np.argpartition(-scores, kth=k-1)[:k]
        part = part[np.argsort(-scores[part])]
        idx_topk = part

    if mode in ("threshold", "both"):
        idx_thr = idx_all[scores >= thr]

    if mode == "topk":
        chosen = idx_topk
    elif mode == "threshold":
        chosen = idx_thr
    else:
        chosen = np.intersect1d(idx_topk, idx_thr, assume_unique=False)

    if chosen is None or len(chosen) == 0:
        return df.iloc[[]].copy()

    return df.iloc[chosen].sort_values("Score", ascending=False).reset_index(drop=True)

def build_tf_tg_indices(df_edges: pd.DataFrame) -> Tuple[Dict[str, int], Dict[str, int]]:
    tf_names = sorted(df_edges["tf_name"].unique().tolist())
    tg_names = sorted(df_edges["tg_name"].unique().tolist())
    tf2idx = {n: i for i, n in enumerate(tf_names)}
    tg2idx = {n: i for i, n in enumerate(tg_names)}
    return tf2idx, tg2idx

def plot_topic_cellcluster_heatmap(H: np.ndarray, clusters: List[int], fname: str):
    with np.errstate(divide="ignore", invalid="ignore"):
        row_sum = H.sum(axis=1, keepdims=True)
        H_prop = np.divide(H, np.maximum(row_sum, 1e-12))

    xt = [f"C{cid}" for cid in clusters]
    yt = [f"T{k}" for k in range(H_prop.shape[0])]

    fig_w = max(10, 1.2 * len(xt)) + 2.0
    fig_h = max(8, 0.8 * len(yt))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(H_prop, aspect="auto", cmap=HEATMAP_CMAP)
    ax.set_title("Topic Analysis", pad=20)
    ax.set_xticks(np.arange(len(xt)))
    ax.set_xticklabels(xt, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(yt)))
    ax.set_yticklabels(yt)

    from mpl_toolkits.axes_grid1 import make_axes_locatable
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Proportion", labelpad=13)

    fig.subplots_adjust(bottom=0.2, right=0.92, top=0.9)
    if SAVE_FIG:
        fig.savefig(os.path.join(OUTPUT_DIR, fname), dpi=220)
    plt.show()

# ======= Step 1: read edges =======
if CLUSTERS is None:
    CLUSTERS = discover_clusters(NETWORK_DIR)
assert CLUSTERS, f"No C{{cid}}_network.* files found in {NETWORK_DIR}"

# trajectory order
ordered_clusters = [c for c in DESIRED_ORDER if c in CLUSTERS]
remaining = [c for c in CLUSTERS if c not in ordered_clusters]
CLUSTERS = ordered_clusters + remaining
print("Using clusters ordered by trajectory:", CLUSTERS)

records = []
for cid in CLUSTERS:
    df_net = load_network_for_cluster(cid, NETWORK_DIR)
    df_sel = select_edges_df(df_net, mode=SELECTION_MODE, top_n=TOP_N_PER_CLUSTER, thr=SCORE_THRESHOLD)
    if df_sel.empty:
        print(f"[Warn] C{cid}: no edges after selection, skip.")
        continue
    for _, r in df_sel.iterrows():
        records.append({
            "cell_cluster": int(cid),
            "tf_name": str(r["TF"]),
            "tg_name": str(r["TG"]),
            "score": float(r["Score"])
        })

df_edges = pd.DataFrame(records)
if df_edges.empty:
    raise RuntimeError("No edges collected from network/; check filters or NETWORK_DIR.")

# ======= Step 2: build edge×cluster matrix =======
tf2idx, tg2idx = build_tf_tg_indices(df_edges)
tg_idx2name = {i: n for n, i in tg2idx.items()}

df_edges["tf_idx"] = df_edges["tf_name"].map(tf2idx).astype(int)
df_edges["tg_idx"] = df_edges["tg_name"].map(tg2idx).astype(int)
df_edges["cell_cluster"] = df_edges["cell_cluster"].astype(int)

C_df = (df_edges.assign(val=1)
        .pivot_table(index=["tf_idx", "tg_idx"],
                     columns="cell_cluster",
                     values="val",
                     aggfunc="sum",
                     fill_value=0))

# ensure columns in CLUSTERS order
for c in CLUSTERS:
    if c not in C_df.columns:
        C_df[c] = 0
C_df = C_df[CLUSTERS]

edge_keys = C_df.index
C_sparse = csr_matrix(C_df.to_numpy(dtype=np.float64))

print(f"Edges after merge: {len(df_edges)}, unique edges: {C_sparse.shape[0]}, clusters: {C_sparse.shape[1]}")

# ======= Step 3: LDA =======
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    learning_method=LDA_LEARNING_METHOD,
    max_iter=LDA_MAX_ITER,
    random_state=RANDOM_STATE,
    evaluate_every=-1,
    verbose=0,
    n_jobs=-1,
)

W = lda.fit_transform(C_sparse)  # (n_edges × n_topics)
H = lda.components_              # (n_topics × n_clusters)

# ======= Step 4: heatmap =======
plot_topic_cellcluster_heatmap(H, CLUSTERS, "topic_x_cellCluster_heatmap_lda_from_network.png")

# ======= Save artifacts for Part B =======
art_path = os.path.join(OUTPUT_DIR, "lda_artifacts.pkl")
with open(art_path, "wb") as f:
    pickle.dump(
        {
            "W": W.astype(np.float32),
            "H": H.astype(np.float32),
            "edge_keys": edge_keys,
            "CLUSTERS": CLUSTERS,
            "tg_idx2name": tg_idx2name,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(f"[Part A done] Heatmap saved. Artifacts -> {art_path}")

In [ ]:
# =========================
# Part B: Enrichment only (grid bars for all topics)
# =========================
# -*- coding: utf-8 -*-

import os, math, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======= output directory and parameters of part A =======
OUTPUT_DIR = "result_topic_lda"
SAVE_FIG = True

ENRICH_DB = "GO_Biological_Process_2021"
TOPK_EDGES_PER_TOPIC_FOR_ENRICH = 500
TOPK_TERMS_PER_TOPIC_BAR = 10

# ======= topic-cluster association in the titles: threshold 10%, at most 3 =======
TOPIC_CLUSTER_THR = 0.10
TOPIC_CLUSTER_TOPN = 3
TOPIC_CLUSTER_DIGITS = 3  # three decimals, e.g. 0.245

# ======= load the intermediate results of part A =======
art_path = os.path.join(OUTPUT_DIR, "lda_artifacts.pkl")
with open(art_path, "rb") as f:
    art = pickle.load(f)

W = art["W"]
H = art["H"]
edge_keys = art["edge_keys"]
CLUSTERS = art["CLUSTERS"]
tg_idx2name = art["tg_idx2name"]

# ======= utils =======
def topic_top_edges(W: np.ndarray, k: int):
    idx = np.argsort(-W, axis=0)  # descending within each column (topic)
    return [idx[:k, t] for t in range(W.shape[1])]

def _topic_cluster_suffixes_from_H(H, clusters, thr=0.10, topn=3, digits=3):
    """topic -> ' | C4(0.55), C1(0.245)'"""
    H = np.asarray(H, dtype=float)
    row_sum = H.sum(axis=1, keepdims=True)
    H_prop = H / np.maximum(row_sum, 1e-12)

    out = {}
    for t in range(H_prop.shape[0]):
        props = H_prop[t]
        order = np.argsort(-props)
        pairs = [(clusters[i], float(props[i])) for i in order]

        sel = [(c, p) for c, p in pairs if p >= thr]
        if len(sel) == 0:
            sel = pairs[:1]          # if none reaches the threshold, show the top one
        else:
            sel = sel[:topn]         # at most topn (= 3)

        parts = [f"C{c}({p:.{digits}f})" for c, p in sel]
        out[t] = " | " + ", ".join(parts)
    return out

def prep_enrichment_bar_df(enrich_df: pd.DataFrame, top_terms: int):
    if enrich_df is None or enrich_df.empty:
        return pd.DataFrame()
    if "Adjusted P-value" not in enrich_df.columns or "Term" not in enrich_df.columns:
        return pd.DataFrame()
    dfp = enrich_df.copy()
    dfp["score"] = -np.log10(np.maximum(dfp["Adjusted P-value"].astype(float), 1e-300))
    dfp = dfp.sort_values("score", ascending=True).tail(top_terms)
    return dfp

def draw_enrichment_bars_on_ax(dfp: pd.DataFrame, ax: plt.Axes, title: str):
    if dfp is None or dfp.empty:
        ax.text(0.5, 0.5, "No enrichment", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return
    ax.barh(np.arange(len(dfp)), dfp["score"].to_numpy())
    ax.set_yticks(np.arange(len(dfp)))
    ax.set_yticklabels(dfp["Term"].tolist())
    ax.set_xlabel("-log10(adj p)")
    ax.set_title(title, pad=10)

    ax.grid(axis="x", linestyle="--", alpha=0.3)

    # slightly smaller fonts in the grid so that panels do not overlap
    ax.tick_params(axis="both", labelsize=13)
    ax.xaxis.label.set_size(16)

def draw_all_topics_enrichment_grid(enrich_dict, n_topics, out_png, db_name):
    # title suffix: Topic0 | C4(0.55), ...
    t2suffix = _topic_cluster_suffixes_from_H(
        H, CLUSTERS,
        thr=TOPIC_CLUSTER_THR,
        topn=TOPIC_CLUSTER_TOPN,
        digits=TOPIC_CLUSTER_DIGITS
    )

    ncols = 2
    nrows = math.ceil(n_topics / ncols)

    # larger canvas and spacing to avoid overlap
    fig_w = 28
    fig_h = max(10, 7.5 * nrows)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_w, fig_h))
    axes = np.array(axes).reshape(-1)

    for t in range(n_topics):
        ax = axes[t]
        df_enr = enrich_dict.get(t, pd.DataFrame())
        dfp = prep_enrichment_bar_df(df_enr, top_terms=TOPK_TERMS_PER_TOPIC_BAR)

        # title format: Topic0 | C4(0.55), C1(0.245)
        title = f"Topic{t}{t2suffix.get(t,'')}"
        draw_enrichment_bars_on_ax(dfp, ax, title)

    for k in range(n_topics, len(axes)):
        axes[k].set_axis_off()

    fig.suptitle(f"Enrichment results for all topics — {db_name}", fontsize=26, y=0.995)

    # wider column and row spacing to avoid overlap
    fig.subplots_adjust(
        left=0.34, right=0.985,
        top=0.94, bottom=0.06,
        wspace=2,  # space between columns
        hspace=0.4   # space between rows
    )

    if SAVE_FIG:
        fig.savefig(os.path.join(OUTPUT_DIR, out_png), dpi=220)
    plt.show()

def run_enrichment_per_topic(W: np.ndarray, edge_keys: pd.MultiIndex, db: str):
    edge_list = list(edge_keys)

    def tg_name_from_idx(idx: int) -> str:
        return tg_idx2name.get(idx, f"TG{idx}")

    try:
        import gseapy as gp
    except Exception as e:
        print("gseapy is not available; skipping the enrichment analysis.", e)
        return

    top_idx_per_topic = topic_top_edges(W, k=min(TOPK_EDGES_PER_TOPIC_FOR_ENRICH, W.shape[0]))
    enrich_dict = {}

    for t, idxs in enumerate(top_idx_per_topic):
        tg_ids = sorted(set(edge_list[i][1] for i in idxs))
        gene_list = [tg_name_from_idx(i) for i in tg_ids]

        if len(gene_list) < 3:
            print(f"[Topic{t}] gene list too small ({len(gene_list)}), skip.")
            enrich_dict[t] = pd.DataFrame()
            continue

        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=[db],
                outdir=None,
                no_plot=True,
                cutoff=1.0
            )
            if isinstance(enr.results, pd.DataFrame):
                df_enr = enr.results.copy()
            else:
                df_enr = list(enr.results.values())[0].copy()
        except Exception as e:
            print(f"[Topic{t}] enrich failed: {e}")
            enrich_dict[t] = pd.DataFrame()
            continue

        enrich_dict[t] = df_enr

        out_csv = os.path.join(OUTPUT_DIR, f"topic_T{t}_enrich_{db}_from_network.csv")
        df_enr.to_csv(out_csv, index=False)

    draw_all_topics_enrichment_grid(
        enrich_dict=enrich_dict,
        n_topics=W.shape[1],
        out_png=f"all_topics_enrich_{db}_from_network.png",
        db_name=db
    )

# ======= run =======
run_enrichment_per_topic(W, edge_keys, db=ENRICH_DB)
print("[Part B done] Enrichment grid + per-topic csv saved.")